# NovelForge - Jalon 3 & 4 : Baseline ML et Evaluation

Objectif : construire une baseline Machine Learning classique pour predire plusieurs genres a partir du texte nettoye d'un Light Novel / Manhwa.

Le modele utilise un pipeline modulaire `TF-IDF + LogisticRegression regularisee L2` encapsule dans `src/baseline_ml.py`.

## 1. Imports et configuration

Le notebook garde uniquement l'orchestration de l'experience. La logique metier reste dans `src/`.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from baseline_ml import BaselineModel
from preprocessing import (
    TextPreprocessor,
    add_filtered_label_column,
    clean_dataframe,
    drop_columns_if_present,
    infer_column,
    parse_multilabel_cell,
    remove_synopsis_anomalies,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

PROJECT_DIR

WindowsPath('C:/Users/ClémentPERRET/OneDrive - EQUATERRE-VDS/Bureau/Cours/DeepLearning')

## 2. Chargement du dataset nettoye

On charge le dataset Manga/Manhwa/Manhua depuis `data/data.csv`, on supprime les colonnes techniques, puis on reconstruit le nettoyage textuel avec les fonctions du Jalon 2.

In [2]:
def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl")
    raise ValueError(f"Unsupported dataset format: {suffix}")


raw_candidates = [
    Path(os.getenv("NOVELFORGE_DATASET", "")) if os.getenv("NOVELFORGE_DATASET") else None,
    PROJECT_DIR / "data" / "data.csv",
    PROJECT_DIR / "data" / "dataset.csv",
    PROJECT_DIR / "data" / "novelforge.csv",
]

raw_path = next((path for path in raw_candidates if path is not None and path.exists()), None)
if raw_path is None:
    raise FileNotFoundError("No raw dataset found in data/.")

df_raw = load_dataset(raw_path)
colonnes_a_supprimer = ["cover"]
df_raw = drop_columns_if_present(df_raw, colonnes_a_supprimer)

text_column = infer_column(
    df_raw.columns,
    candidates=["synopsis", "summary", "description", "overview", "plot", "resume"],
)
genre_column = infer_column(
    df_raw.columns,
    candidates=["genres", "genre", "tags", "categories", "labels", "target"],
)

GENRE_VOCABULARY = [
    "Action",
    "Adventure",
    "Comedy",
    "Drama",
    "Fantasy",
    "Romance",
    "School Life",
    "Slice of Life",
    "Supernatural",
    "Mystery",
    "Psychological",
    "Horror",
    "Historical",
    "Sci Fi",
    "Sports",
    "Martial Arts",
    "Magic",
    "Isekai",
    "Harem",
    "Mecha",
    "Demons",
    "Seinen",
    "Shoujo",
    "Shounen",
    "Josei",
    "BL",
    "GL",
    "Yaoi",
    "Yuri",
    "Shounen-ai",
    "Shoujo-ai",
    "Smut",
    "Wuxia",
    "Xianxia",
]

df_raw = add_filtered_label_column(
    df_raw,
    label_column=genre_column,
    output_column="genre_labels",
    allowed_labels=GENRE_VOCABULARY,
)

preprocessor = TextPreprocessor(lowercase=True, remove_urls=True, lemmatize=True)
df = clean_dataframe(df_raw, text_column=text_column, clean_column="synopsis_clean", preprocessor=preprocessor)
df = remove_synopsis_anomalies(df, clean_column="synopsis_clean", min_words=5)
df = df[df["genre_labels"].str.len().gt(0)].copy()

print(f"Raw dataset loaded and cleaned on the fly: {raw_path}")
print(f"Columns removed if present: {colonnes_a_supprimer}")
print(f"Text column: {text_column}")
print(f"Raw label column: {genre_column}")
print(f"Shape after cleaning: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(df.head())

Raw dataset loaded and cleaned on the fly: C:\Users\ClémentPERRET\OneDrive - EQUATERRE-VDS\Bureau\Cours\DeepLearning\data\data.csv
Columns removed if present: ['cover']
Text column: description
Raw label column: tags
Shape after cleaning: 69,511 rows x 10 columns


,title,description,rating,year,tags,genre_labels,synopsis_clean,word_count,char_count,raw_char_count
0,Salad Days (Tang LiuZang) - Part 2,The second season of Salad Days (Tang LiuZang).,4.7,2021.0,"['BL', 'Manhua', 'Romance', 'Shounen-ai', 'Sports', 'Webtoons', 'Full Color']","[BL, Romance, Shounen-ai, Sports]",the second season of salad day tang liuzang,8,43,47
1,The Master of Diabolism,"As the grandmaster who founded the Demonic Sect, Wei WuXian roamed the world in his wanton ways, hated by millions for the chaos he crea...",4.7,2017.0,"['Action', 'Adventure', 'BL', 'Comedy', 'Manhua', 'Mystery', 'Romance', 'Shounen-ai', 'Ancient China', 'Cultivation', 'Martial Arts', 'S...","[Action, Adventure, BL, Comedy, Mystery, Romance, Shounen-ai, Martial Arts, Supernatural, Xianxia]",as the grandmaster who found the demonic sect wei wuxian roam the world in his wanton way hat by million for the chao he creat in the en...,76,397,429
2,JoJo's Bizarre Adventure Part 7: Steel Ball Run,"Set in 1890, Steel Ball Run spotlights Gyro Zepelli and Johnny Joestar as they pit their spirits on a Fifty Million Dollar race across t...",4.7,2004.0,"['Action', 'Adventure', 'Horror', 'Mystery', 'Seinen', '19th Century', 'America', 'Historical']","[Action, Adventure, Horror, Mystery, Seinen, Historical]",set in steel ball run spotlight gyro zepelli and johnny joestar as they pit their spirit on a fifty million dollar race across the heart...,69,363,384
3,A Sign of Affection,"Yuki is a typical college student, whose world revolves around her friends, social media, and the latest sales. But when a chance encoun...",4.7,2019.0,"['Romance', 'Shoujo', 'Slice of Life', 'Disability']","[Romance, Shoujo, Slice of Life]",yuki is a typical college student whose world revolve around her friend social media and the latest sale but when a chance encounter on ...,66,383,405
4,Moriarty the Patriot,"Before he was Sherlock’s rival, Moriarty fought against the unfair class caste system in London by making sure corrupt nobility got thei...",4.7,2016.0,"['Mystery', 'Shounen', 'Detectives', 'England', 'Europe', 'Historical', 'Sherlock Holmes', 'Adapted to Anime', 'Based on a Novel']","[Mystery, Shounen, Historical]",before he was sherlock s rival moriarty fought against the unfair class caste system in london by mak sure corrupt nobility got their co...,53,289,298


## 3. Preparation des labels multilabel

Les tags sont filtres dans une taxonomie de genres, puis transformes en matrice binaire avec `MultiLabelBinarizer`. Les genres trop rares sont filtres pour eviter des classes peu fiables dans l'evaluation.

In [3]:
TEXT_COLUMN = "synopsis_clean"
LABEL_COLUMN = "genre_labels"

# Safety parsing for notebooks reloaded from CSV exports where lists may become strings.
df_model = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna(subset=[TEXT_COLUMN]).copy()
df_model[LABEL_COLUMN] = df_model[LABEL_COLUMN].apply(parse_multilabel_cell)
df_model = df_model[df_model[LABEL_COLUMN].str.len().gt(0)].copy()

genre_counts = df_model[LABEL_COLUMN].explode().value_counts()
MIN_GENRE_FREQ = 50
kept_genres = set(genre_counts[genre_counts >= MIN_GENRE_FREQ].index)

df_model[LABEL_COLUMN] = df_model[LABEL_COLUMN].apply(lambda genres: [genre for genre in genres if genre in kept_genres])
df_model = df_model[df_model[LABEL_COLUMN].str.len().gt(0)].copy()

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df_model[LABEL_COLUMN])
X = df_model[TEXT_COLUMN].fillna("").astype(str)

print(f"Text column: {TEXT_COLUMN}")
print(f"Filtered label column: {LABEL_COLUMN}")
print(f"Rows kept for ML: {len(df_model):,}")
print(f"Genres kept with at least {MIN_GENRE_FREQ} examples: {len(mlb.classes_):,}")
display(genre_counts.head(20).to_frame("count"))

Text column: synopsis_clean
Filtered label column: genre_labels
Rows kept for ML: 69,511
Genres kept with at least 50 examples: 34


,count
genre_labels,
Romance,29762
Comedy,21282
Drama,18702
Fantasy,16125
Action,12734
BL,12667
School Life,12582
Yaoi,10131
Seinen,9235


## 4. Split train/test

La classification multilabel rend la stratification classique difficile. On utilise donc un split aleatoire reproductible et on controle ensuite les scores micro/macro pour surveiller les classes rares.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
)

print(f"Train size: {len(X_train):,}")
print(f"Test size:  {len(X_test):,}")
print(f"Label matrix shape: {Y.shape}")

Train size: 55,608
Test size:  13,903
Label matrix shape: (69511, 34)


## 5. Entrainement de la baseline regularisee

Le modele utilise une regression logistique `One-vs-Rest` : un classifieur binaire est appris pour chaque genre. La regularisation L2 est active via `penalty="l2"` et controlee par le parametre `C`.

In [5]:
baseline = BaselineModel(
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1_000,
    random_state=42,
)

baseline.train(X_train, y_train)
print("Baseline trained.")

Baseline trained.


## 6. Evaluation multilabel

L'accuracy classique est peu informative en multilabel : elle peut etre tres severe ou trompeuse selon le nombre de genres par oeuvre. On privilegie donc les F1 micro et macro.

In [6]:
metrics = baseline.evaluate(
    X_test,
    y_test,
    target_names=list(mlb.classes_),
    X_train=X_train,
    y_train=y_train,
)

summary_metrics = pd.Series(
    {
        "train_f1_micro": metrics["train_f1_micro"],
        "test_f1_micro": metrics["test_f1_micro"],
        "generalization_gap_micro": metrics["generalization_gap_micro"],
        "train_f1_macro": metrics["train_f1_macro"],
        "test_f1_macro": metrics["test_f1_macro"],
        "generalization_gap_macro": metrics["generalization_gap_macro"],
        "test_f1_weighted": metrics["test_f1_weighted"],
        "test_jaccard_samples": metrics["test_jaccard_samples"],
        "test_hamming_loss": metrics["test_hamming_loss"],
    }
).to_frame("score")

display(summary_metrics.round(4))
print(metrics["classification_report_text"])

,score
train_f1_micro,0.5517
test_f1_micro,0.4213
generalization_gap_micro,0.1304
train_f1_macro,0.5632
test_f1_macro,0.3734
generalization_gap_macro,0.1898
test_f1_weighted,0.4895
test_jaccard_samples,0.2849
test_hamming_loss,0.1393


               precision    recall  f1-score   support

       Action       0.56      0.58      0.57      2514
    Adventure       0.40      0.59      0.47      1267
           BL       0.64      0.69      0.66      2576
       Comedy       0.51      0.43      0.46      4239
       Demons       0.41      0.57      0.47       244
        Drama       0.42      0.57      0.48      3687
      Fantasy       0.64      0.55      0.59      3179
           GL       0.23      0.40      0.29       388
        Harem       0.21      0.39      0.28       328
   Historical       0.34      0.46      0.39       765
       Horror       0.22      0.39      0.28       398
       Isekai       0.41      0.70      0.52       325
        Josei       0.37      0.52      0.43      1080
        Magic       0.32      0.56      0.41       353
 Martial Arts       0.26      0.52      0.35       249
        Mecha       0.02      0.74      0.04       120
      Mystery       0.28      0.43      0.34       697
Psycholog

## 7. Lecture du rapport par genre

Le F1 macro donne le meme poids a chaque genre : il est donc sensible aux genres rares. Le F1 micro agrege toutes les decisions label par label : il reflete davantage la performance globale sur les genres frequents.

In [7]:
report_df = pd.DataFrame(metrics["classification_report"]).T
per_genre_report = report_df.loc[mlb.classes_, ["precision", "recall", "f1-score", "support"]]

display(per_genre_report.sort_values("support", ascending=False).head(20).round(3))
display(per_genre_report.sort_values("f1-score", ascending=True).head(20).round(3))

,precision,recall,f1-score,support
Romance,0.702,0.594,0.644,5912.0
Comedy,0.505,0.429,0.464,4239.0
Drama,0.417,0.568,0.481,3687.0
Fantasy,0.641,0.546,0.590,3179.0
BL,0.644,0.686,0.664,2576.0
Action,0.561,0.583,0.571,2514.0
School Life,0.556,0.577,0.567,2504.0
Yaoi,0.583,0.664,0.621,2063.0
Seinen,0.197,0.714,0.309,1820.0
Shoujo,0.363,0.533,0.432,1693.0


,precision,recall,f1-score,support
Mecha,0.020,0.742,0.038,120.0
Yuri,0.021,0.555,0.040,173.0
Sports,0.049,0.822,0.092,276.0
Wuxia,0.250,0.062,0.100,16.0
Shoujo-ai,0.163,0.347,0.222,216.0
Psychological,0.168,0.348,0.226,477.0
Smut,0.210,0.370,0.268,270.0
Shounen-ai,0.186,0.505,0.272,513.0
Harem,0.213,0.390,0.275,328.0
Horror,0.224,0.387,0.284,398.0


## 8. Regularisation et compromis Biais/Variance

Le pipeline utilise une regression logistique regularisee L2, aussi appelee regularisation Ridge. Sur une representation TF-IDF, le nombre de variables peut devenir tres grand : chaque mot ou n-gramme devient une feature. Sans regularisation, le modele peut donner des poids excessifs a des termes rares presents dans le train, ce qui augmente le risque de surapprentissage.

La regularisation L2 penalise les grands coefficients. Concretement, elle force le modele a repartir le poids sur plusieurs indices textuels plutot que de memoriser quelques tokens tres specifiques. Le parametre `C` controle cette contrainte : plus `C` est petit, plus la regularisation est forte; plus `C` est grand, plus le modele est libre d'ajuster les poids.

Pour le compromis biais/variance, on compare les scores train et test. Si les F1 train sont eleves mais les F1 test nettement plus faibles, le modele a une variance trop forte et surapprend. Si les F1 train et test sont tous les deux faibles, le modele est probablement en sous-apprentissage : TF-IDF + regression logistique ne capte pas assez la semantique, ou la regularisation est trop forte. Un ecart raisonnable entre train et test avec un F1 test correct indique une baseline stable, utile comme reference avant de passer a des modeles deep learning.